In [1]:
print("SpiderNet started 🕸️")
import pandas as pd
import numpy as np

SpiderNet started 🕸️


In [2]:
df = pd.read_csv("../data/train.csv")   # replace with your file name
df.head()

,Dates,Category,Descript,DayOfWeek,PdDistrict,Resolution,Address,X,Y
0,2015-05-13 23:53:00,WARRANTS,WARRANT ARREST,Wednesday,NORTHERN,"ARREST, BOOKED",OAK ST / LAGUNA ST,-122.425892,37.774599
1,2015-05-13 23:53:00,OTHER OFFENSES,TRAFFIC VIOLATION ARREST,Wednesday,NORTHERN,"ARREST, BOOKED",OAK ST / LAGUNA ST,-122.425892,37.774599
2,2015-05-13 23:33:00,OTHER OFFENSES,TRAFFIC VIOLATION ARREST,Wednesday,NORTHERN,"ARREST, BOOKED",VANNESS AV / GREENWICH ST,-122.424363,37.800414
3,2015-05-13 23:30:00,LARCENY/THEFT,GRAND THEFT FROM LOCKED AUTO,Wednesday,NORTHERN,NONE,1500 Block of LOMBARD ST,-122.426995,37.800873
4,2015-05-13 23:30:00,LARCENY/THEFT,GRAND THEFT FROM LOCKED AUTO,Wednesday,PARK,NONE,100 Block of BRODERICK ST,-122.438738,37.771541


In [3]:
df.columns

Index(['Dates', 'Category', 'Descript', 'DayOfWeek', 'PdDistrict',
       'Resolution', 'Address', 'X', 'Y'],
      dtype='object')

In [4]:
df_clean = df[["Dates", "Category", "X", "Y"]]
df_clean.head()

,Dates,Category,X,Y
0,2015-05-13 23:53:00,WARRANTS,-122.425892,37.774599
1,2015-05-13 23:53:00,OTHER OFFENSES,-122.425892,37.774599
2,2015-05-13 23:33:00,OTHER OFFENSES,-122.424363,37.800414
3,2015-05-13 23:30:00,LARCENY/THEFT,-122.426995,37.800873
4,2015-05-13 23:30:00,LARCENY/THEFT,-122.438738,37.771541


In [5]:
df_clean = df_clean.rename(columns={
    "Dates": "date",
    "Category": "crime_type",
    "X": "longitude",
    "Y": "latitude"
})

df_clean.head()

,date,crime_type,longitude,latitude
0,2015-05-13 23:53:00,WARRANTS,-122.425892,37.774599
1,2015-05-13 23:53:00,OTHER OFFENSES,-122.425892,37.774599
2,2015-05-13 23:33:00,OTHER OFFENSES,-122.424363,37.800414
3,2015-05-13 23:30:00,LARCENY/THEFT,-122.426995,37.800873
4,2015-05-13 23:30:00,LARCENY/THEFT,-122.438738,37.771541


In [6]:
df_clean["date"] = pd.to_datetime(df_clean["date"])
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 878049 entries, 0 to 878048
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   date        878049 non-null  datetime64[ns]
 1   crime_type  878049 non-null  object        
 2   longitude   878049 non-null  float64       
 3   latitude    878049 non-null  float64       
dtypes: datetime64[ns](1), float64(2), object(1)
memory usage: 26.8+ MB


In [7]:
df_clean = df_clean.sort_values("date")
df_clean.head()

,date,crime_type,longitude,latitude
878048,2003-01-06 00:01:00,FORGERY/COUNTERFEITING,-122.394926,37.738212
878045,2003-01-06 00:01:00,LARCENY/THEFT,-122.447364,37.731948
878047,2003-01-06 00:01:00,VANDALISM,-122.390531,37.780607
878046,2003-01-06 00:01:00,LARCENY/THEFT,-122.403390,37.780266
878044,2003-01-06 00:15:00,ROBBERY,-122.459033,37.714056


In [8]:
df_clean["week"] = df_clean["date"].dt.to_period("W")
df_clean.head()

,date,crime_type,longitude,latitude,week
878048,2003-01-06 00:01:00,FORGERY/COUNTERFEITING,-122.394926,37.738212,2003-01-06/2003-01-12
878045,2003-01-06 00:01:00,LARCENY/THEFT,-122.447364,37.731948,2003-01-06/2003-01-12
878047,2003-01-06 00:01:00,VANDALISM,-122.390531,37.780607,2003-01-06/2003-01-12
878046,2003-01-06 00:01:00,LARCENY/THEFT,-122.403390,37.780266,2003-01-06/2003-01-12
878044,2003-01-06 00:15:00,ROBBERY,-122.459033,37.714056,2003-01-06/2003-01-12


In [9]:
weekly_counts = (
    df_clean
    .groupby(["latitude", "longitude", "week"])
    .size()
    .reset_index(name="crime_count")
)

weekly_counts.head()

,latitude,longitude,week,crime_count
0,37.707879,-122.463626,2007-02-05/2007-02-11,1
1,37.707920,-122.460921,2004-12-06/2004-12-12,2
2,37.707920,-122.460921,2006-12-11/2006-12-17,1
3,37.707920,-122.460921,2008-06-23/2008-06-29,1
4,37.707922,-122.428717,2003-05-12/2003-05-18,1


In [10]:
weekly_counts = weekly_counts.sort_values(
    ["latitude", "longitude", "week"]
)

In [11]:
weekly_counts = weekly_counts.sort_values(
    ["latitude", "longitude", "week"]
)

In [12]:
weekly_counts["previous_week"] = (
    weekly_counts
    .groupby(["latitude", "longitude"])["crime_count"]
    .shift(1)
)
weekly_counts["percent_increase"] = (
    (weekly_counts["crime_count"] - weekly_counts["previous_week"])
    / weekly_counts["previous_week"]
) * 100
weekly_counts["percent_increase"] = (
    weekly_counts["percent_increase"]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)
weekly_counts["spike"] = weekly_counts["percent_increase"] > 50

weekly_counts.head()

,latitude,longitude,week,crime_count,previous_week,percent_increase,spike
0,37.707879,-122.463626,2007-02-05/2007-02-11,1,NaN,0.0,False
1,37.707920,-122.460921,2004-12-06/2004-12-12,2,NaN,0.0,False
2,37.707920,-122.460921,2006-12-11/2006-12-17,1,2.0,-50.0,False
3,37.707920,-122.460921,2008-06-23/2008-06-29,1,1.0,0.0,False
4,37.707922,-122.428717,2003-05-12/2003-05-18,1,NaN,0.0,False


In [13]:
spikes = weekly_counts[weekly_counts["spike"] == True]
spikes.head()

,latitude,longitude,week,crime_count,previous_week,percent_increase,spike
6,37.707922,-122.428717,2005-05-16/2005-05-22,2,1.0,100.0,True
10,37.707922,-122.428717,2013-11-25/2013-12-01,2,1.0,100.0,True
15,37.707968,-122.463545,2008-03-17/2008-03-23,2,1.0,100.0,True
21,37.708003,-122.460832,2012-03-19/2012-03-25,2,1.0,100.0,True
25,37.708031,-122.428737,2006-01-23/2006-01-29,2,1.0,100.0,True


In [14]:
spikes.shape

(100829, 7)

In [15]:
spikes.columns

Index(['latitude', 'longitude', 'week', 'crime_count', 'previous_week',
       'percent_increase', 'spike'],
      dtype='object')

In [16]:
weekly_counts["risk_score"] = (
    weekly_counts["crime_count"] *
    (1 + weekly_counts["percent_increase"] / 100)
)

In [17]:
spikes = weekly_counts[weekly_counts["spike"] == True]
spikes.head()

,latitude,longitude,week,crime_count,previous_week,percent_increase,spike,risk_score
6,37.707922,-122.428717,2005-05-16/2005-05-22,2,1.0,100.0,True,4.0
10,37.707922,-122.428717,2013-11-25/2013-12-01,2,1.0,100.0,True,4.0
15,37.707968,-122.463545,2008-03-17/2008-03-23,2,1.0,100.0,True,4.0
21,37.708003,-122.460832,2012-03-19/2012-03-25,2,1.0,100.0,True,4.0
25,37.708031,-122.428737,2006-01-23/2006-01-29,2,1.0,100.0,True,4.0


In [18]:
spikes.shape

(100829, 8)

In [19]:
import folium

In [21]:
sf_map = folium.Map(location=[37.77, -122.42], zoom_start=12)

In [ ]:
for _, row in spikes.iterrows():
    folium.Circle(
        location=[row["latitude"], row["longitude"]],
        radius=row["risk_score"] * 0.05,   # smaller scaling
        popup=f"""
        Week: {row['week']}<br>
        Crime Count: {row['crime_count']}<br>
        Increase: {round(row['percent_increase'],2)}%<br>
        Risk Score: {round(row['risk_score'],2)}
        """,
        color="red",
        fill=True,
        fill_opacity=0.5
    ).add_to(sf_map)

sf_map